# G37 — Frames for the case-study / motivation figure (no model, no training)

Renders frames of clips I–IV for the MCIS selected in `analysis/case_study_select.py` from the **saved** test
predictions (final model: G36 `Up-sqrt`; baseline: G10) (descriptive use, decided by the authors). Clip IV frames are shown only as illustration of the target
turn; no model reads them. Two images per MCIS: a 4 × 3 grid (three frames per clip) and a one-row strip (middle frame
of each clip) for the paper. Inputs: `hi-ef-dataset` only. CPU is enough.

In [ ]:
import os, glob, textwrap
import numpy as np, pandas as pd, cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
OUT_DIR = "/kaggle/working/g37_frames"
os.makedirs(OUT_DIR, exist_ok=True)
PICK = [{'sample_id': 'sample02416', 'category': 'win_shift', 'clip1': '47/00532', 'clip2': '47/00533', 'clip3': '47/00534', 'clip4': '47/00535', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'disgust', 'Baseline': 'happy', 'p_true_RoleNet': 0.492, 'p_true_Baseline': 0.104}, {'sample_id': 'sample01031', 'category': 'win_shift', 'clip1': '15/00639', 'clip2': '15/00640', 'clip3': '15/00641', 'clip4': '15/00642', 'A_III': 'surprise', 'B_IV': 'disgust', 'RoleNet': 'disgust', 'Baseline': 'angry', 'p_true_RoleNet': 0.485, 'p_true_Baseline': 0.154}, {'sample_id': 'sample01043', 'category': 'win_shift', 'clip1': '15/00790', 'clip2': '15/00791', 'clip3': '15/00792', 'clip4': '15/00793', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'disgust', 'Baseline': 'happy', 'p_true_RoleNet': 0.366, 'p_true_Baseline': 0.051}, {'sample_id': 'sample00225', 'category': 'win_shift', 'clip1': '02/00647', 'clip2': '02/00648', 'clip3': '02/00649', 'clip4': '02/00650', 'A_III': 'surprise', 'B_IV': 'neutral', 'RoleNet': 'neutral', 'Baseline': 'angry', 'p_true_RoleNet': 0.44, 'p_true_Baseline': 0.153}, {'sample_id': 'sample01312', 'category': 'win_mirror', 'clip1': '21/00078', 'clip2': '21/00079', 'clip3': '21/00080', 'clip4': '21/00081', 'A_III': 'sad', 'B_IV': 'sad', 'RoleNet': 'sad', 'Baseline': 'happy', 'p_true_RoleNet': 0.582, 'p_true_Baseline': 0.251}, {'sample_id': 'sample01305', 'category': 'win_mirror', 'clip1': '21/00047', 'clip2': '21/00048', 'clip3': '21/00049', 'clip4': '21/00050', 'A_III': 'happy', 'B_IV': 'happy', 'RoleNet': 'happy', 'Baseline': 'neutral', 'p_true_RoleNet': 0.44, 'p_true_Baseline': 0.139}, {'sample_id': 'sample01344', 'category': 'win_mirror', 'clip1': '21/00558', 'clip2': '21/00559', 'clip3': '21/00560', 'clip4': '21/00561', 'A_III': 'angry', 'B_IV': 'angry', 'RoleNet': 'angry', 'Baseline': 'happy', 'p_true_RoleNet': 0.473, 'p_true_Baseline': 0.198}, {'sample_id': 'sample01306', 'category': 'fail_shift_rare', 'clip1': '21/00049', 'clip2': '21/00050', 'clip3': '21/00051', 'clip4': '21/00052', 'A_III': 'happy', 'B_IV': 'surprise', 'RoleNet': 'happy', 'Baseline': 'happy', 'p_true_RoleNet': 0.048, 'p_true_Baseline': 0.114}, {'sample_id': 'sample02762', 'category': 'fail_shift_rare', 'clip1': '53/00184', 'clip2': '53/00185', 'clip3': '53/00186', 'clip4': '53/00187', 'A_III': 'angry', 'B_IV': 'disgust', 'RoleNet': 'sad', 'Baseline': 'angry', 'p_true_RoleNet': 0.09, 'p_true_Baseline': 0.155}, {'sample_id': 'sample01345', 'category': 'fail_shift_rare', 'clip1': '21/00569', 'clip2': '21/00570', 'clip3': '21/00571', 'clip4': '21/00572', 'A_III': 'angry', 'B_IV': 'surprise', 'RoleNet': 'angry', 'Baseline': 'angry', 'p_true_RoleNet': 0.067, 'p_true_Baseline': 0.131}]
roots = sorted(glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF")))
VIDEO_ROOTS = [os.path.join(r, "video") for r in roots if os.path.isdir(os.path.join(r, "video"))]
ANNOT_CSV = [os.path.join(r, "annotation.csv") for r in roots if os.path.exists(os.path.join(r, "annotation.csv"))][0]
ann = pd.read_csv(ANNOT_CSV, header=None, dtype=str).set_index(0)
print("video roots:", VIDEO_ROOTS, "| MCIS to render:", len(PICK))


def video_path(clip):
    ep, num = clip.split('/')
    for root in VIDEO_ROOTS:
        for ext in ('.mp4', '.avi', '.mkv', '.mov'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


def frames(clip, n=3):
    p = video_path(clip)
    if p is None:
        return [np.zeros((180, 320, 3), np.uint8)] * n
    cap = cv2.VideoCapture(p)
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    out = []
    for t in np.linspace(0.15, 0.85, n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * (tot - 1)))
        ok, fr = cap.read()
        out.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if ok else np.zeros((180, 320, 3), np.uint8))
    cap.release()
    return out


def text(c):
    t = ann.at[c, 1] if c in ann.index else ''
    return t if isinstance(t, str) else ''


for s in PICK:
    clips = [s['clip1'], s['clip2'], s['clip3'], s['clip4']]
    F = [frames(c) for c in clips]
    names = ['I (context)', 'II (context)', f"III (A: {s['A_III']})", f"IV (target B: {s['B_IV']})"]
    fig, ax = plt.subplots(3, 4, figsize=(13, 6.2))
    for k in range(4):
        for j in range(3):
            ax[j, k].imshow(F[k][j]); ax[j, k].axis('off')
        ax[0, k].set_title(names[k] + "\n" + "\n".join(textwrap.wrap('"' + text(clips[k]) + '"', 38)[:2]), fontsize=8)
    fig.suptitle(f"{s['sample_id']} [{s['category']}]  RoleNet: {s['RoleNet']} (p_true {s['p_true_RoleNet']})   "
                 f"baseline: {s['Baseline']} (p_true {s['p_true_Baseline']})", fontsize=9)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{s['sample_id']}_grid.png", dpi=130); plt.close(fig)
    fig, ax = plt.subplots(1, 4, figsize=(13, 2.6))
    for k in range(4):
        ax[k].imshow(F[k][1]); ax[k].axis('off')
        ax[k].set_title(names[k] + "\n" + "\n".join(textwrap.wrap('"' + text(clips[k]) + '"', 40)[:2]), fontsize=8)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{s['sample_id']}_strip.png", dpi=200); plt.close(fig)
    print("rendered", s['sample_id'], s['category'])
pd.DataFrame(PICK).to_csv(f"{OUT_DIR}/picked.csv", index=False)
print("saved to", OUT_DIR)